In [16]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("KMeansClustering").getOrCreate()


In [5]:
df = spark.read.csv("hdfs://localhost:9000/user/yash1122/DataSet/Clustering_ds/Mall_Customers.csv", header=True, inferSchema=True)
df.show(50)


+----------+------+---+------------------+----------------------+
|CustomerID| Genre|Age|Annual Income (k$)|Spending Score (1-100)|
+----------+------+---+------------------+----------------------+
|         1|  Male| 19|                15|                    39|
|         2|  Male| 21|                15|                    81|
|         3|Female| 20|                16|                     6|
|         4|Female| 23|                16|                    77|
|         5|Female| 31|                17|                    40|
|         6|Female| 22|                17|                    76|
|         7|Female| 35|                18|                     6|
|         8|Female| 23|                18|                    94|
|         9|  Male| 64|                19|                     3|
|        10|Female| 30|                19|                    72|
|        11|  Male| 67|                19|                    14|
|        12|Female| 35|                19|                    99|
|        1

In [6]:
data = df.select("Age", "Annual Income (k$)", "Spending Score (1-100)")

In [7]:
from pyspark.ml.feature import VectorAssembler

assembler = VectorAssembler(
    inputCols=["Age", "Annual Income (k$)", "Spending Score (1-100)"],
    outputCol="features"
)

final_data = assembler.transform(data)


In [23]:
from pyspark.ml.clustering import KMeans

kmeans = KMeans(featuresCol="features", k=3, seed=1)
model = kmeans.fit(final_data)


In [25]:
predictions = model.transform(final_data)
predictions.show(50)

+---+------------------+----------------------+----------------+----------+
|Age|Annual Income (k$)|Spending Score (1-100)|        features|prediction|
+---+------------------+----------------------+----------------+----------+
| 19|                15|                    39|[19.0,15.0,39.0]|         0|
| 21|                15|                    81|[21.0,15.0,81.0]|         0|
| 20|                16|                     6| [20.0,16.0,6.0]|         1|
| 23|                16|                    77|[23.0,16.0,77.0]|         0|
| 31|                17|                    40|[31.0,17.0,40.0]|         0|
| 22|                17|                    76|[22.0,17.0,76.0]|         0|
| 35|                18|                     6| [35.0,18.0,6.0]|         1|
| 23|                18|                    94|[23.0,18.0,94.0]|         0|
| 64|                19|                     3| [64.0,19.0,3.0]|         1|
| 30|                19|                    72|[30.0,19.0,72.0]|         0|
| 67|       

In [10]:
from pyspark.ml.evaluation import ClusteringEvaluator

evaluator = ClusteringEvaluator()
silhouette = evaluator.evaluate(predictions)
print(f"Silhouette Score = {silhouette}")

Silhouette Score = 0.45097978411312134


In [11]:
centers = model.clusterCenters()
for idx, center in enumerate(centers):
    print(f"Cluster {idx} Center: {center}")


Cluster 0 Center: [26.5952381  33.14285714 65.66666667]
Cluster 1 Center: [45.30769231 60.52991453 34.47008547]
Cluster 2 Center: [32.97560976 88.73170732 79.24390244]
